### Proyecto 3: actas de reunión desde audio

Pipeline de dos etapas, cada una con el modelo que le corresponde:

1. **Transcripción (ASR)** con un modelo abierto (Whisper vía `transformers.pipeline`). Corre local, no gasta cuota de API — la tarea es mecánica (audio → texto), no necesita razonamiento.
2. **Redacción del acta** con un modelo frontier (Gemini) a partir del texto crudo transcripto. Acá sí importa entender contexto: resumir, sacar decisiones y action items de una transcripción desordenada (sin puntuación de discurso, con muletillas).

Mismo principio que en semana 1/2: local para lo mecánico, frontier para lo que requiere razonar.

#### Paso 0 (placeholder): generar un audio de prueba

No hay una grabación de reunión real a mano todavía, así que generamos un audio corto con el TTS de Gemini (mismo patrón que en `02-apis-streaming-chatbots/codigo/03-multimodal-tts.ipynb`) simulando 2-3 líneas de una reunión. Cuando haya un audio real, este paso se salta y se apunta directo al archivo.

In [1]:
from dotenv import load_dotenv
from google import genai
from google.genai import types
import os
import wave

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

guion_reunion = (
    "Bueno, arrancamos. El tema de hoy es el lanzamiento del proyecto. "
    "Juan va a terminar el informe de presupuesto para el viernes. "
    "Maria queda a cargo de coordinar con el cliente la fecha de entrega. "
    "Quedamos en juntarnos de nuevo el lunes que viene para revisar avances."
)

response = client.models.generate_content(
    model="gemini-2.5-flash-preview-tts",
    contents=guion_reunion,
    config=types.GenerateContentConfig(
        response_modalities=["AUDIO"],
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Kore")
            )
        ),
    ),
)
audio_part = response.candidates[0].content.parts[0].inline_data

with wave.open("reunion_prueba.wav", "wb") as wav_file:
    wav_file.setnchannels(1)
    wav_file.setsampwidth(2)
    wav_file.setframerate(24000)
    wav_file.writeframes(audio_part.data)

print("Audio de prueba generado: reunion_prueba.wav")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Audio de prueba generado: reunion_prueba.wav


#### Paso 1: transcripción con Whisper (modelo abierto, local)

`openai/whisper-small` corre bien en la GPU que ya tenemos configurada. El pipeline de `transformers` se encarga de resamplear el audio, chunkear si es largo, y devolver el texto plano.

In [2]:
import torch
from transformers import pipeline

transcriptor = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=0 if torch.cuda.is_available() else -1,
)

resultado = transcriptor("reunion_prueba.wav", generate_kwargs={"language": "spanish"})
transcripcion = resultado["text"]
print(transcripcion)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 Bueno, arrancamos. El tema de hoy es el lanzamiento del proyecto. Juan va a terminar el informe de presupuesto para el viernes. María queda a cargo de coordinar con el cliente la fecha de entrega. Quedamos en juntarnos de nuevo el lunes que viene para revisar avances.


#### Paso 2: redactar el acta con Gemini (frontier)

Le pasamos la transcripción cruda y le pedimos estructura: resumen, decisiones, action items con responsable.

In [3]:
prompt_acta = f"""Sos un asistente que redacta actas de reunión a partir de una transcripción cruda de audio (puede tener errores de transcripción, sin puntuación clara).

Transcripción:
\"\"\"{transcripcion}\"\"\"

Redactá un acta con esta estructura:
1. Resumen breve (2-3 líneas)
2. Decisiones tomadas
3. Action items (responsable + tarea)
"""

response = client.models.generate_content(
    model="gemini-flash-lite-latest",
    contents=prompt_acta,
)
print(response.text)

**ACTA DE REUNIÓN**

**1. Resumen breve**
Reunión inicial destinada a coordinar los preparativos para el lanzamiento del proyecto. Se definieron las responsabilidades clave para la gestión del presupuesto y la coordinación con el cliente, estableciendo una próxima fecha de revisión para el lunes siguiente.

**2. Decisiones tomadas**
* Se coordinará la fecha de entrega directamente con el cliente.
* Se fijó una nueva reunión de seguimiento para el próximo lunes para revisar los avances del proyecto.

**3. Action items**
* **Juan:** Terminar el informe de presupuesto (Fecha límite: viernes).
* **María:** Coordinar con el cliente la fecha de entrega.
